# Phase 3: Geospatial Assignment and Spatial Joins

In this notebook we implement the geospatial assignment. We take the raw synthetic citizen telemetry that we created with the open geographic data (OGD) layers from the City of Vienna (including district administrative boundaries, pedestrian zones, and cycle path networks).

The spatial queries  and nearest-neighbor distance metrhing are done to bind each telemetry record to its respective path and infrastrucure context. With this, we export the aggregated throughput counts and static choropleth maps.

*Note*: As the Heterogeneity Human Activity Recognition (HHAR) dataset lacks GPS telemetry, coordinates are generated programmatically using a constant-velocity local flat-earth approximation. Trajectories represent linear paths rather than actual road-network routing.

In [1]:
import os
import sys
import logging
from pathlib import Path

# Climb up from the notebook's folder to find the true project workspace root
notebook_dir = Path(os.getcwd())
PROJECT_ROOT = notebook_dir.parent if (notebook_dir.parent / "src").exists() else notebook_dir

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.bootstrapping import setup_winutils
from pyspark.sql import SparkSession

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Configure Windows-specific local Spark settings dynamically (exactly like notebook 1)
if os.name == 'nt':
    setup_winutils(PROJECT_ROOT)
    os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

spark_builder = (
    SparkSession.builder
    .appName("Vienna-District-Assignment")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "4")
)

if os.name == "nt":
    spark_builder = (
        spark_builder
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.pyspark.python", sys.executable)
        .config("spark.pyspark.driver.python", sys.executable)
    )

spark = spark_builder.getOrCreate()
logging.info(f"Project root: {PROJECT_ROOT}")


2026-07-08 00:21:00,645 - INFO - Hadoop environment path configuration active: HADOOP_HOME=c:\Users\fedka\Documents\GitHub\Geospatial Repletion & Saturation Modelling\data\winutils
2026-07-08 00:21:06,216 - INFO - Project root: c:\Users\fedka\Documents\GitHub\Geospatial Repletion & Saturation Modelling


## 3.1 Input Telemetry Ingestion

The ingestion stage loads the synthesized citizen telemetry from the compressed, columnar Parquet database:
*   **Path**: `data/processed/synthetic_telemetry.parquet`
*   **Fallback**: If the master synthetic telemetry is missing, we create the minimal telemetry generator (`create_minimal_telemetry`) and produce a local diagnostic dataset containing 30 synthetic agents.

In [2]:
from src.create_minimal_telemetry import create_minimal_telemetry

telemetry_path = PROJECT_ROOT / "data" / "processed" / "synthetic_telemetry.parquet"

if not telemetry_path.exists():
    create_minimal_telemetry(PROJECT_ROOT, num_agents=30, samples_per_agent=500)
    spark.catalog.clearCache()

telemetry_df = spark.read.parquet(str(telemetry_path))
print(f"Telemetry rows: {telemetry_df.count()}")
telemetry_df.groupBy("Activity").count().show()

Telemetry rows: 500000
+--------+------+
|Activity| count|
+--------+------+
|    walk|200000|
|    bike|150000|
|   stand|150000|
+--------+------+



## 3.2 Geospatial Assignment & Spatial R-Tree Indexing

In order to resolve the spatial context for the stream, the coordinates we have are mapped against the Vienna OGD layers. The algorithm utilises the R-tree (which is highly optimised spatial indexing) via **Shapely STRtree** objects on the driver. This is needed for the execution of point-in-polygon and distance-to-curve queries:

1.  **Administrative District Matching**: Telemetry coordinates are evaluated against the district boundaries GeoDataFrame using a standard point-in-polygon inclusion test:
    $$\mathbf{p} = (\lambda, \phi) \in \mathcal{P}_{\text{district}}$$
2.  **Pedestrian Zone Assignment (Walking)**: For records classified as `walk` the pedestrian zone R-tree index (`STRtree`) is taken for a coordinate querry against it. This is needed to identify the candidate ovealpping polygons to then return specified pedestrian zone label. 
3.  **Cycle Path Mapping (Biking)**: For records classified as `bike`, the coordinates are matched against the bike path R-tree index. The algorithm calculates the nearest segment distance in degrees, converts it to meters, and binds the record to the cycle path if the distance is within the "tolerance" limit:
    $$d_{\text{metres}} = d_{\text{degrees}} \cdot M_{\text{lat}} \le 150 \text{ meters}$$
    where $M_{\text{lat}} = 111,320.0 \text{ m/degree}$.

In [3]:
from src.geospatial_districts import run_district_assignment

outputs = run_district_assignment(spark, PROJECT_ROOT)
outputs

2026-07-08 00:21:24,559 - INFO - Layer already cached: vienna_districts.geojson
2026-07-08 00:21:24,561 - INFO - Layer already cached: vienna_pedestrian_zones.geojson
2026-07-08 00:21:24,563 - INFO - Layer already cached: vienna_bike_paths.geojson
2026-07-08 00:21:26,929 - INFO - Loading cached street graph: vienna_walk_network.graphml
2026-07-08 00:22:46,638 - INFO - Collecting telemetry data to driver for spatial context assignment...
2026-07-08 00:22:53,236 - INFO - Building spatial search indices on the driver...
2026-07-08 00:22:53,258 - INFO - Assigning spatial contexts to 500000 records...
2026-07-08 00:23:48,689 - INFO - Writing district summary CSV...
2026-07-08 00:23:48,701 - INFO - Writing infrastructure summary CSV...
2026-07-08 00:23:49,212 - INFO - Saved district choropleth map to c:\Users\fedka\Documents\GitHub\Geospatial Repletion & Saturation Modelling\data\geospatial_output\district_choropleth.png
2026-07-08 00:23:49,759 - INFO - Saved scatter map to c:\Users\fedka\Do

{'summary_csv': WindowsPath('c:/Users/fedka/Documents/GitHub/Geospatial Repletion & Saturation Modelling/data/geospatial_output/district_activity_counts.csv'),
 'infrastructure_csv': WindowsPath('c:/Users/fedka/Documents/GitHub/Geospatial Repletion & Saturation Modelling/data/geospatial_output/infrastructure_activity_counts.csv'),
 'map_png': WindowsPath('c:/Users/fedka/Documents/GitHub/Geospatial Repletion & Saturation Modelling/data/geospatial_output/district_choropleth.png'),
 'scatter_png': WindowsPath('c:/Users/fedka/Documents/GitHub/Geospatial Repletion & Saturation Modelling/data/geospatial_output/telemetry_scatter_map.png'),
 'playback_html': WindowsPath('c:/Users/fedka/Documents/GitHub/Geospatial Repletion & Saturation Modelling/data/geospatial_output/interactive_citizens_map.html'),
 'districts_geojson': WindowsPath('c:/Users/fedka/Documents/GitHub/Geospatial Repletion & Saturation Modelling/data/spatial/vienna_districts.geojson'),
 'pedestrian_geojson': WindowsPath('c:/Users

In [4]:
summary_df = spark.read.csv(
    f"{PROJECT_ROOT}/data/geospatial_output/district_activity_counts.csv",
    header=True,
    inferSchema=True
)
print(f"District summary rows: {summary_df.count()}")
print(f"Unique districts: {summary_df.select('district_name').distinct().count()}")

from pyspark.sql.functions import sum as spark_sum, col
district_totals = (
    summary_df.groupBy("district_name", "district_number")
    .agg(spark_sum("count").alias("simulated_record_count"))
    .orderBy(col("simulated_record_count").desc())
)
print("\nTop districts by simulated record count:")
district_totals.show(12)


District summary rows: 53
Unique districts: 20

Top districts by simulated record count:
+-------------+---------------+----------------------+
|district_name|district_number|simulated_record_count|
+-------------+---------------+----------------------+
|    Favoriten|             10|                140000|
|   Donaustadt|             22|                 80000|
| Leopoldstadt|             02|                 50000|
|      Liesing|             23|                 39800|
| Innere Stadt|             01|                 32800|
|    Simmering|             11|                 30000|
|   Josefstadt|             08|                 10000|
|      Währing|             18|                 10000|
|    Mariahilf|             06|                 10000|
|      Penzing|             14|                 10000|
|  Brigittenau|             20|                 10000|
|       Neubau|             07|                 10000|
+-------------+---------------+----------------------+
only showing top 12 rows



## 3.3 Data Sources & Spatial Provenance

*   **Sensor Telemetry Source**: UCI Heterogeneity Human Activity Recognition (HHAR) Dataset (licensed under CC BY 4.0).
*   **Spatial Reference Layers**: City of Vienna Open Government Data (OGD) (licensed under CC BY 4.0 AT), including Bezirksgrenzen (districts), Fußgängerzonen (pedestrian zones), and Radwege (bike paths).
*   **Pipeline Scope**: All coordinate data points are synthetic. The output counts and visualizations serve as a proof-of-concept validation of the Spark and spatial indexing pipeline; they do not represent actual mobility, foot traffic, or transit congestion levels in Vienna.

In [5]:
spark.stop()

## 3.4 Interactive Real-time Telemetry Playback App

The following interactive Leaflet visualization displays our synthetic citizens moving along their street routes in real-time. 

*   **Controls**: Use the play/pause button, the scrubbing slider, or the speed control bar at the bottom left to play back the simulation.
*   **Legend**: Walking citizens are styled in **green**, biking citizens in **blue**, and stationary citizens in **purple**.

In [6]:
from IPython.display import IFrame
IFrame(src="../data/geospatial_output/interactive_citizens_map.html", width="100%", height=600)